# 27 — AWS Few-Shot: EPISODE-Level Split + Bootstrap 95% CI

**Dijalankan di SageMaker.** Robustness check yang lebih tepat daripada temporal-cut naif (nb26).

**Latar (temuan investigasi nb26):** timeline Fase 2 menaruh SEMUA serangan di menit 1–5 lalu
menit 6–15 murni benign. Akibatnya potong-waktu 60/40 membuat train 92% attack dan test 76%
benign — komposisi kelas timpang antar-blok, bukan ukuran generalisasi episode yang bermakna.

**Desain episode-split (B1):** split berdasarkan **jenis episode serangan** (`gt_category`),
bukan potong-waktu. Benign dibagi proporsional ke train & test agar KEDUA sisi punya attack + benign:
- **TRAIN** = episode {BruteForce, DoS} + 50% benign
- **TEST**  = episode {DDoS} (tak terlihat saat kalibrasi) + 50% benign sisanya

Pertanyaan yang dijawab (sesuai reviewer): *apakah kalibrasi few-shot dari episode teramati
(BruteForce/DoS) membantu mendeteksi episode BERBEDA (DDoS) yang tak dilihat saat kalibrasi?*
Tidak ada flow episode-yang-sama di kedua sisi — menghilangkan kekhawatiran within-episode memorization.

> Input: `detect_clean_flows.csv` (+ volumetric bila ada) dari S3 `unsw-far/results/` (butuh `gt_category`).
> Output: `episode_split_out/aws_episode_results.json` + kurva + CSV -> S3 `unsw-far/episode/`.
> Semua angka dari eksekusi nyata; bila hasil kurang mendukung, dilaporkan apa adanya.

In [ ]:
import importlib, sys, subprocess
pkgmap={'sklearn':'scikit-learn'}
need=[m for m in ('xgboost','sklearn','pandas','numpy','matplotlib','boto3') if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*[pkgmap.get(m,m) for m in need]],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score
import xgboost as xgb
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3})

S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='episode_split_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; N_BOOT=1000
# Definisi episode split (bisa diubah bila ingin arah lain, mis. DDoS ke train)
TRAIN_EPISODES=['BruteForce','DoS']; TEST_EPISODES=['DDoS']
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','seed':SEED,'features':CANON,
         'design':'episode-level split by gt_category; benign split 50/50; bootstrap 95% CI for MCC',
         'train_episodes':TRAIN_EPISODES,'test_episodes':TEST_EPISODES,'n_boot':N_BOOT}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def metrics(y,yp): return dict(mcc=round(float(matthews_corrcoef(y,yp)),4),f1=round(float(f1_score(y,yp,zero_division=0)),4),precision=round(float(precision_score(y,yp,zero_division=0)),4),recall=round(float(recall_score(y,yp,zero_division=0)),4),n=int(len(y)),n_pos=int(np.sum(y)))

def boot_ci_mcc(y_true,y_pred,n_boot=N_BOOT,seed=SEED,alpha=0.05):
    rng=np.random.default_rng(seed); y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); N=len(y_true); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,N,N); yt,yp=y_true[idx],y_pred[idx]
        if len(np.unique(yt))<2: continue
        vals.append(matthews_corrcoef(yt,yp))
    if not vals: return (float('nan'),float('nan'),float('nan'))
    lo,hi=np.percentile(vals,[100*alpha/2,100*(1-alpha/2)])
    return (round(float(np.mean(vals)),4),round(float(lo),4),round(float(hi),4))
print('=== SEL 1 (import & konfigurasi) SELESAI ===')

## 1. Rakit dataset AWS berlabel + `gt_category` (dari S3, selalu fresh)

In [ ]:
AWS_DIR='aws_labeled'; os.makedirs(AWS_DIR,exist_ok=True)
want=['detect_clean_flows.csv','detect_volumetric_flows.csv']
try:
    import boto3; s3=boto3.client('s3',region_name=REGION)
    for fn in want:
        try: s3.download_file(S3_BUCKET,f'{S3_PREFIX}/results/{fn}',os.path.join(AWS_DIR,fn)); print('  download (fresh):',fn)
        except Exception as e: print('  (lewati)',fn,e)
except Exception as e: print('download S3 dilewati:',e)

parts=[]
for fn in want:
    p=os.path.join(AWS_DIR,fn)
    if os.path.exists(p):
        d=pd.read_csv(p); print(f'  {fn}: kolom={list(d.columns)}')
        if 'gt_category' not in d.columns: print(f'  !! {fn} tanpa gt_category -> tak bisa episode-split.'); continue
        if 'ground_truth' in d.columns and all(c in d.columns for c in CANON):
            keep=CANON+['ground_truth','gt_category']+(['elapsed_sec'] if 'elapsed_sec' in d.columns else [])
            dd=d[keep].rename(columns={'ground_truth':'y'}); dd['src_file']=fn; parts.append(dd)
    else: print(f'  {fn}: TIDAK ADA di {AWS_DIR}/')
assert parts, 'Tidak ada CSV ber-gt_category. Cek header CSV di atas / download S3.'
aws=pd.concat(parts,ignore_index=True).replace([np.inf,-np.inf],np.nan).dropna(subset=CANON+['y'])
aws['y']=aws['y'].astype(int)
print('AWS berlabel:',len(aws),'| y:',aws['y'].value_counts().to_dict())
print('kategori:',aws['gt_category'].value_counts().to_dict())
RESULTS['aws_total']={'n':int(len(aws)),'n_attack':int(aws['y'].sum()),'n_benign':int((aws['y']==0).sum()),
                      'by_category':{k:int(v) for k,v in aws['gt_category'].value_counts().items()}}
aws.to_csv(os.path.join(OUTDIR,'aws_labeled_all.csv'),index=False)
print('=== SEL 2 (rakit dataset + gt_category) SELESAI ===')

## 2. EPISODE split: train={BruteForce,DoS}+benign, test={DDoS}+benign
Benign dibagi 50/50 (acak, seed tetap) agar kedua sisi punya kelas negatif. Serangan dipisah
berdasarkan jenis episode — test hanya berisi DDoS yang TIDAK ada di train.

In [ ]:
rng=np.random.default_rng(SEED)
benign=aws[aws['gt_category']=='Benign'].copy()
atk_tr=aws[aws['gt_category'].isin(TRAIN_EPISODES)].copy()
atk_te=aws[aws['gt_category'].isin(TEST_EPISODES)].copy()
# bagi benign 50/50
perm=rng.permutation(len(benign)); half=len(benign)//2
ben_tr=benign.iloc[perm[:half]]; ben_te=benign.iloc[perm[half:]]
aws_tr=pd.concat([atk_tr,ben_tr],ignore_index=True)
aws_te=pd.concat([atk_te,ben_te],ignore_index=True)
print('TRAIN episodes',TRAIN_EPISODES,'-> n=',len(aws_tr),aws_tr['y'].value_counts().to_dict())
print('  kategori TRAIN:',aws_tr['gt_category'].value_counts().to_dict())
print('TEST  episodes',TEST_EPISODES,'-> n=',len(aws_te),aws_te['y'].value_counts().to_dict())
print('  kategori TEST :',aws_te['gt_category'].value_counts().to_dict())
assert aws_te['y'].nunique()>1, 'TEST hanya 1 kelas -> MCC tak terdefinisi. Tinjau pembagian benign/episode.'
assert aws_tr['y'].nunique()>1, 'TRAIN hanya 1 kelas.'
Xte,yte=aws_te[CANON].values, aws_te['y'].values
RESULTS['episode_split']={'train':int(len(aws_tr)),'test':int(len(aws_te)),
  'train_pos':int(aws_tr['y'].sum()),'train_neg':int((aws_tr['y']==0).sum()),
  'test_pos':int(aws_te['y'].sum()),'test_neg':int((aws_te['y']==0).sum())}
aws_tr.to_csv(os.path.join(OUTDIR,'aws_episode_train.csv'),index=False)
aws_te.to_csv(os.path.join(OUTDIR,'aws_episode_test.csv'),index=False)
print('=== SEL 3 (episode split) SELESAI ===')

## 3. Muat SOURCE (UNS/CIC 9-fitur) — identik konvensi nb22/nb26

In [ ]:
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
def load_uns():
    f=first(['../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])
    if not f: return None
    u=pd.read_csv(f); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','label']
    if any(c not in u.columns for c in need): return None
    X=pd.DataFrame({'duration':pd.to_numeric(u['dur'],errors='coerce')*1e6,'fwd_pkts':u['spkts'],'bwd_pkts':u['dpkts'],
                    'fwd_bytes':u['sbytes'],'bwd_bytes':u['dbytes'],'fwd_mean':u['smean'],'bwd_mean':u['dmean'],
                    'src_load':pd.to_numeric(u['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u['dpkts'],errors='coerce')/pd.to_numeric(u['dur'],errors='coerce').replace(0,np.nan)})
    X['y']=(pd.to_numeric(u['label'],errors='coerce')>0).astype(int)
    return X.replace([np.inf,-np.inf],np.nan).dropna()[CANON+['y']]
def load_cic():
    f=first(['../../CICDDoS2018/data/cleaned_100.pkl','../../CICDDoS2018/data/cleaned_*.pkl'])
    if not f: return None
    import pickle
    with open(f,'rb') as fh: d=pickle.load(fh)
    feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
    Xo=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
    cdf=pd.DataFrame(Xo,columns=feats)
    cmap={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
          'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    if any(v not in cdf.columns for v in cmap.values()): return None
    out=pd.DataFrame({k:cdf[cmap[k]].values for k in CANON})
    yraw=np.asarray(d['y']) if 'y' in d else np.asarray(d.get('labels'))
    out['y']=(pd.to_numeric(pd.Series(yraw),errors='coerce').fillna(0)>0).astype(int).values
    return out[CANON+['y']].replace([np.inf,-np.inf],np.nan).dropna()
SOURCES={}
for name,loader in [('UNS',load_uns),('CIC',load_cic)]:
    s=loader()
    if s is not None and len(s)>0: SOURCES[name]=s; print(f'SOURCE {name}: n={len(s)} {s["y"].value_counts().to_dict()}')
    else: print(f'SOURCE {name}: TIDAK tersedia')
RESULTS['sources_available']=list(SOURCES.keys())
assert SOURCES, 'Tidak ada source.'
print('=== SEL 4 (muat SOURCE) SELESAI ===')

## 4. Zero-shot + Few-shot (episode) + Bootstrap 95% CI

In [ ]:
def train_xgb(X,y,seed=SEED):
    y=np.asarray(y); n_pos=int((y==1).sum()); n_neg=int((y==0).sum())
    params=dict(max_depth=8,learning_rate=0.1,n_estimators=300,subsample=0.9,colsample_bytree=0.9,eval_metric='logloss',n_jobs=-1,random_state=seed)
    if n_pos>0 and n_neg>0: params['scale_pos_weight']=n_neg/n_pos
    clf=xgb.XGBClassifier(**params); clf.fit(X,y); return clf
def sample_frac(df,frac,rng):
    if frac>=1.0: return df
    outs=[]
    for c,g in df.groupby('y'):
        n=max(1,int(round(len(g)*frac))); outs.append(g.iloc[rng.permutation(len(g))[:n]])
    return pd.concat(outs)
def run_source(name,src):
    rng=np.random.default_rng(SEED); rows=[]
    m=train_xgb(src[CANON].values,src['y'].values); yp=m.predict(Xte)
    r=metrics(yte,yp); ci=boot_ci_mcc(yte,yp)
    r.update(source=name,k_percent=0.0,mode=f'zero-shot {name}',mcc_ci_lo=ci[1],mcc_ci_hi=ci[2],mcc_boot_mean=ci[0])
    rows.append(r); print(f'[{name}] zero-shot MCC={r["mcc"]} CI=[{ci[1]},{ci[2]}] P={r["precision"]} R={r["recall"]}')
    for k in [1,5,10,20,50]:
        aws_k=sample_frac(aws_tr,k/100.0,rng)
        aws_k.to_csv(os.path.join(OUTDIR,f'aws_episode_fewshot_{name}_{k}pct.csv'),index=False)
        Xtr=np.vstack([src[CANON].values,aws_k[CANON].values]); ytr=np.r_[src['y'].values,aws_k['y'].values]
        m=train_xgb(Xtr,ytr); yp=m.predict(Xte); r=metrics(yte,yp); ci=boot_ci_mcc(yte,yp)
        r.update(source=name,k_percent=float(k),mode=f'few-shot {name}+{k}%AWS',n_aws_train=int(len(aws_k)),mcc_ci_lo=ci[1],mcc_ci_hi=ci[2],mcc_boot_mean=ci[0])
        rows.append(r); print(f'[{name}] few-shot {k}% (n={len(aws_k)}) MCC={r["mcc"]} CI=[{ci[1]},{ci[2]}] P={r["precision"]} R={r["recall"]}')
    return rows
res_rows=[]
for name,src in SOURCES.items(): res_rows+=run_source(name,src)
print('=== SEL 5 (zero-shot + few-shot episode + CI) SELESAI ===')

## 5. Acuan atas: AWS-only (train episode {BruteForce,DoS}+benign, test DDoS+benign)

In [ ]:
m=train_xgb(aws_tr[CANON].values,aws_tr['y'].values); yp=m.predict(Xte)
r=metrics(yte,yp); ci=boot_ci_mcc(yte,yp)
r.update(source='AWS',k_percent=100.0,mode='AWS-only (acuan atas)',mcc_ci_lo=ci[1],mcc_ci_hi=ci[2],mcc_boot_mean=ci[0])
res_rows.append(r); print('AWS-only:',{kk:r[kk] for kk in ['mcc','mcc_ci_lo','mcc_ci_hi','precision','recall']})
RESULTS['runs']=res_rows
tab=pd.DataFrame(res_rows)[['source','mode','k_percent','mcc','mcc_ci_lo','mcc_ci_hi','f1','precision','recall','n_pos','n']]
import IPython.display as ipd; ipd.display(tab)
tab.to_csv(os.path.join(OUTDIR,'aws_episode_table.csv'),index=False)
print('=== SEL 6 (AWS-only + tabel + CI) SELESAI ===')

## 6. Kurva few-shot dengan pita 95% CI + simpan + UPLOAD S3

In [ ]:
fig,ax=plt.subplots(figsize=(7.4,4.4)); colmap={'UNS':'#DD8452','CIC':'#4C72B0'}
for name in RESULTS.get('sources_available',[]):
    pts=[r for r in res_rows if r.get('source')==name and (r['k_percent']==0.0 or r['mode'].startswith('few-shot'))]
    pts=sorted(pts,key=lambda r:r['k_percent'])
    xs=[r['k_percent'] for r in pts]; ys=[r['mcc'] for r in pts]; lo=[r['mcc_ci_lo'] for r in pts]; hi=[r['mcc_ci_hi'] for r in pts]
    ax.plot(xs,ys,'o-',lw=2,color=colmap.get(name),label=f'{name}+AWS (episode)')
    ax.fill_between(xs,lo,hi,alpha=0.18,color=colmap.get(name))
aws_only=next((r['mcc'] for r in res_rows if r['mode'].startswith('AWS-only')),None)
if aws_only is not None: ax.axhline(aws_only,ls='--',color='#55A868',label=f'AWS-only={aws_only:.3f}')
ax.set_xlabel('% AWS labels (from train episodes: %s)'%('+'.join(TRAIN_EPISODES)))
ax.set_ylabel('MCC on unseen episode (%s)+benign'%('+'.join(TEST_EPISODES)))
ax.set_ylim(-0.3,1.0); ax.set_title('Few-shot on AWS — EPISODE split (unseen DDoS test, 95% CI)'); ax.legend()
plt.tight_layout(); savefig('aws_episode_fewshot_curve.png')
json_path=os.path.join(OUTDIR,'aws_episode_results.json')
with open(json_path,'w') as f: json.dump(RESULTS,f,indent=2)
print('tersimpan',json_path)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/episode/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/episode/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 7 (kurva + simpan + upload) SELESAI ===')
print('SEMUA SELESAI. Bandingkan: zero-shot vs few-shot pada episode DDoS tak-terlihat.')